In [1]:
### Import Libraries
from pathlib import Path
import numpy as np              # ver_1.26
import pandas as pd             # ver_2.1
import re
from utils.Final_medical_test import *
from utils.ml_utils import *

In [2]:
### Data Path
current_path = Path.cwd()          
project_root = current_path

data_path = project_root / "data"
image_path = project_root / "images"

# print("# data path:", data_path)

In [3]:
### Load Data
sample_df = pd.read_csv(data_path / 'SAMPLE_DATA.csv',sep=',', encoding='cp949')
print('# data shape : ', sample_df.shape)

# data shape :  (26, 123)


---

In [4]:
sample_df

,lid,Exam Age,days,AGIS,CIGTS,MD,VFI,TDV_01,TDV_02,TDV_03,...,PDV_45,PDV_46,PDV_47,PDV_48,PDV_49,PDV_50,PDV_51,PDV_52,PDV_53,PDV_54
0,95790,45,0,1,0.09615,-1.47,98,1,1,2,...,-4,-3,-1,-2,-2,0,-2,-1,-2,-4
1,95790,46,263,1,0.00000,-1.65,97,1,1,1,...,-3,-2,0,-1,-2,0,-1,0,-2,0
2,95790,46,507,1,0.00000,-1.80,97,0,1,1,...,-2,-2,-1,-1,-2,-3,-4,-1,-3,-2
3,95790,47,677,1,0.57692,-1.46,97,-3,1,2,...,-1,-2,-1,-4,-2,-4,-3,-1,-2,-2
4,95790,47,938,1,0.38462,-1.41,98,-1,0,1,...,-3,-1,0,-2,-1,-4,0,-2,0,2
5,95790,48,1183,0,0.28846,-1.41,98,-2,1,2,...,-2,-2,-1,-1,-1,-1,-4,-1,0,-3
6,95790,49,1516,1,0.00000,-2.01,96,1,1,-1,...,-1,-1,-1,-2,-2,-3,-3,-1,-1,-2
7,95790,50,1694,1,0.00000,-0.94,97,2,1,1,...,-2,-2,0,0,-2,0,-1,-1,0,0
8,95790,50,1902,0,0.00000,-1.02,98,3,2,0,...,-3,-1,0,0,-1,-3,-3,0,-2,-1
9,95790,51,2121,0,0.00000,-0.74,98,0,1,1,...,-3,0,0,-1,-4,-4,0,1,-1,-3


In [5]:
# x_i = the ith days in sample_df.columns
# Check : x_{i+1} - x_i >= 28
remove_close_days_df = remove_close_days(sample_df, minimum_days)
print('# data shape : ', remove_close_days_df.shape)

# data shape :  (26, 123)


In [6]:
# Split : x_{i+1} - x_i > 730
# Check : len(days) >= 5
# Re-referenced to day 0
divide_df = divide_dataframe(remove_close_days_df, group_period)
print('# data shape : ', divide_df.shape)

# data shape :  (24, 124)


In [7]:
# Check all of progressions
medical_df = medical_test(divide_df, agis_test=True, cigts_test=True, md_slope=True, vfi_slope=True, plr_test=True)
wiggs_df = wiggs_test(medical_df, event_based=True, trend_based_MD=True, trend_based_PDV=True)

In [8]:
# Divide by 5 days
# Re-referenced to day 0
fix_df = fix_length(wiggs_df, length=5)
print('# data shape : ', fix_df.shape)

# data shape :  (20, 134)


In [9]:
# Check : x_5 - x_1 > 730 of each eye_episode
less_df = sublid_less_than_2y(fix_df)
print('# data shape : ', less_df.shape)

# data shape :  (20, 134)


In [10]:
# Make each labels
consensus_df = consensus_deterioration(less_df)
wiggs_df = wiggs_deterioration(consensus_df)

In [11]:
# Difference between the late-phase mean and the early-phase mean (late minus early)
mean_df = mean_diff(wiggs_df)

# Make TDV.mean, TDV.std, PDV.mean, PDV.std
final_df = modi_dataframe(mean_df)

print('# data shape : ', final_df.shape)

# data shape :  (4, 140)


In [12]:
features = ['lid', 'eye_episode', 'Exam Age', 'days', 'AGIS', 'CIGTS', 'MD', 'MD_slope_value', 'VFI', 'VFI_slope_value', 'TDVmean', 'TDVstd', 'PDVmean', 'PDVstd', 'Consensus_label', 'Wiggs_label']

In [13]:
final_df[features]

,lid,eye_episode,Exam Age,days,AGIS,CIGTS,MD,MD_slope_value,VFI,VFI_slope_value,TDVmean,TDVstd,PDVmean,PDVstd,Consensus_label,Wiggs_label
0,95790,9579000,58.2,485.833,1.833,-0.096,-0.113,-0.257367,-0.333,-0.341193,-0.137846,1.337324,-0.400654,1.342306,1,1
1,95790,9579001,55.4,577.000,-0.833,0.208,0.140,-0.257367,0.000,-0.341193,-0.019231,2.266520,-0.246788,2.383794,1,1
2,95790,9579002,52.0,681.167,0.167,0.497,-1.080,-0.257367,0.000,-0.341193,-0.926288,2.403794,0.339712,2.405277,1,1
3,95790,9579003,48.8,643.500,0.167,-0.337,0.087,-0.257367,-1.000,-0.341193,-0.182712,2.719958,-0.532077,2.629439,1,1
